<a href="https://colab.research.google.com/github/jihun0251/korean-spam-classifierML/blob/main/%EC%8A%A4%ED%8C%B8_%EB%B6%84%EB%A5%98_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 설치

In [1]:
!pip install konlpy  # 한국어 형태소 분리를 위한 라이브러리 다운로드

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 36.4 MB/s eta 0:00:00


# 2. 라이브러리 불러오기

In [2]:
import pandas as pd  # 데이터를 표처럼 다루는 도구

# --- 텍스트 처리 & baseline 용 ---
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 변환
from sklearn.model_selection import train_test_split          # 학습/시험 분할
from sklearn.metrics import classification_report, confusion_matrix  # 평가
from konlpy.tag import Okt  # 한국어 형태소 분석기

# --- PyTorch (신경망) ---
import torch                       # PyTorch 본체
import torch.nn as nn              # 신경망 부품(층, 활성함수, 손실함수)
from torch.utils.data import DataLoader, TensorDataset  # 데이터를 batch로 공급

# 3. 데이터 만들기 + Okt 준비

In [3]:
# 스팸 문자 50개 - 다양한 유형
spam_messages = [
    "축하합니다! 100만원에 당첨되셨습니다 지금 클릭 http://bit.ly/win123",
    "[Web발신] 국세청 환급금 조회 안내 http://tax-refund.xyz",
    "택배 배송 주소가 불분명합니다 정보 입력 http://delivery-check.com",
    "엄마 나 폰 고장나서 그런데 이 링크로 인증 좀 해줘 http://help-me.net",
    "고객님 카드가 해외에서 결제되었습니다 본인 아니면 클릭 http://card-stop.co",
    "무료 상품권 증정 이벤트! 지금 신청 http://free-gift.shop",
    "대출 한도 5000만원 승인 완료 지금 확인 http://easy-loan.kr",
    "[긴급] 계정이 정지되었습니다 즉시 인증 http://account-fix.io",
    "당신만을 위한 특별 할인 쿠폰 도착 http://coupon-now.link",
    "건강검진 결과 이상 소견 확인하려면 클릭 http://health-check.vip",
    "[국민은행] 본인인증 필요 아래 링크 접속 http://kb-auth.xyz",
    "경품 추첨에 당첨되셨어요 배송지 입력 http://prize-win.net",
    "스미싱 신고 안내 클릭하여 확인하세요 http://report-fake.co",
    "월 300만원 재택 부업 지금 시작 http://home-job.shop",
    "고객님의 택배가 보관중입니다 http://parcel-hold.xyz",
    "비트코인 투자로 하루 50만원 수익 http://coin-rich.io",
    "[카카오] 비정상 로그인 감지 확인 http://kakao-check.link",
    "신용등급 무료 조회 이벤트 http://credit-free.kr",
    "당첨을 축하드립니다 신세계상품권 50만원 http://sse-gift.net",
    "긴급 연락 바랍니다 아버지 사고 병원비 송금 http://hospital-pay.co",
    "넷플릭스 결제 실패 정보 갱신 필요 http://netflix-fix.xyz",
    "회원님 포인트 소멸 예정 지금 사용 http://point-use.shop",
    "정부지원금 신청 대상자 확인 http://gov-support.io",
    "명품 시계 90% 할인 한정 판매 http://luxury-sale.link",
    "로또 1등 번호 무료 공개 http://lotto-win.kr",
    "고객님 명의로 휴대폰이 개통되었습니다 http://phone-open.net",
    "카드 포인트 현금 전환 마지막 기회 http://card-cash.co",
    "[쿠팡] 미수령 택배 주소 확인 http://coupang-fake.xyz",
    "초대장이 도착했습니다 확인 http://invite-card.shop",
    "귀하의 계좌가 보이스피싱에 연루 http://account-safe.io",
    "성인인증 무료 가입 이벤트 http://adult-free.link",
    "재난지원금 25만원 신청하세요 http://disaster-fund.kr",
    "당신의 사진이 유출되었습니다 확인 http://photo-leak.net",
    "[농협] 보안카드 재발급 안내 http://nh-security.co",
    "급전 필요하신가요 무담보 당일대출 http://quick-money.xyz",
    "프리미엄 회원 무료 체험 신청 http://premium-free.shop",
    "택배기사입니다 부재중 연락주세요 http://delivery-call.io",
    "주식 리딩방 무료 입장 수익률 300% http://stock-room.link",
    "고객센터입니다 환불 처리 정보 입력 http://refund-now.kr",
    "당첨 안내 갤럭시 최신폰 무료 증정 http://galaxy-free.net",
    "법원 출석 통지서 확인 바랍니다 http://court-notice.co",
    "카지노 첫충전 100% 보너스 http://casino-bonus.xyz",
    "건강보험 환급금 조회하세요 http://insurance-back.shop",
    "택배 통관 보류 관세 결제 필요 http://customs-pay.io",
    "유명 쇼핑몰 폐업 정리 90% 세일 http://closing-sale.link",
    "본인인증만 하면 5만원 즉시 지급 http://verify-cash.kr",
    "코인 에어드랍 무료 토큰 받기 http://airdrop-free.net",
    "미납 요금이 있습니다 즉시 납부 http://unpaid-bill.co",
    "귀하는 배심원으로 선정되었습니다 http://jury-select.xyz",
    "한정수량 아이폰 추첨 응모 http://iphone-draw.shop",
]

# 정상 문자 50개 - 일상 대화, 공식 안내
ham_messages = [
    "엄마 나 오늘 저녁 약속 있어서 늦어",
    "내일 회의 10시로 변경됐어요 참고 부탁드려요",
    "고객님 통신요금 할인 이벤트 당첨 안내 고객센터로 문의주세요",
    "택배가 오늘 오후에 도착 예정입니다 부재시 경비실에 맡겨드릴게요",
    "주문하신 상품이 배송 시작되었습니다 마이페이지에서 확인하세요",
    "지훈아 이번 주말에 같이 밥 먹자",
    "결제가 정상적으로 완료되었습니다 이용해주셔서 감사합니다",
    "도서관 대출 도서 반납일이 내일까지입니다",
    "축하해 합격 소식 들었어 정말 잘됐다",
    "이번 달 카드 명세서가 발행되었습니다 앱에서 확인 가능합니다",
    "오늘 점심 뭐 먹을까 메뉴 추천 좀",
    "회의 자료 메일로 보냈으니 확인 부탁해요",
    "엄마 김치 떨어졌는데 좀 보내줄 수 있어?",
    "내일 비 온대 우산 챙겨가",
    "프로젝트 마감 다음주 금요일까지인거 잊지마",
    "생일 축하해 선물은 내가 준비했어",
    "병원 예약 내일 오후 3시인거 기억하지?",
    "엄마 나 시험 잘 봤어 걱정 마",
    "오늘 야근이라 저녁 먼저 먹어",
    "주말에 영화 볼래? 예매할게",
    "택배 잘 받았어 고마워",
    "회사 워크샵 다음달 둘째주로 정해졌어요",
    "할머니 댁에 언제 갈지 정하자",
    "운동 같이 하기로 한거 오늘 맞지?",
    "리포트 다 썼어 한번 봐줄래?",
    "고객님 예약이 정상 접수되었습니다",
    "다음 정기검진은 6개월 후입니다",
    "강의 자료 업로드 했으니 확인하세요",
    "이번 학기 수강신청 잘 됐어?",
    "퇴근하고 마트 들러서 갈게 뭐 필요해?",
    "동아리 모임 이번주 수요일 저녁이야",
    "면접 잘 보고 와 응원할게",
    "주문번호 조회되었습니다 내일 출고 예정입니다",
    "친구야 오랜만이다 언제 한번 보자",
    "엄마 용돈 보내주셔서 감사해요",
    "발표 준비 거의 다 끝났어",
    "내일 아침 일찍 출발하자 7시까지 와",
    "감기 조심하고 푹 쉬어",
    "계약서 검토 후 회신 드리겠습니다",
    "오늘 경기 봤어? 진짜 역전승이었어",
    "도서 연체료가 부과되었으니 반납 바랍니다",
    "새 학기 시간표 나왔어 같이 확인하자",
    "택배 수령 확인 부탁드립니다",
    "주말 잘 보내고 월요일에 보자",
    "엄마 나 집 가는 길이야 곧 도착해",
    "팀 회식 장소 예약 완료했습니다",
    "건강하게 잘 지내고 있지? 보고싶다",
    "과제 제출 확인 메일 받았어요 감사합니다",
    "내일 등산 가는데 같이 갈래?",
    "수고했어 오늘 정말 고생 많았어",
]

import pandas as pd
df = pd.DataFrame({
    "message": spam_messages + ham_messages,
    "label": ["spam"] * len(spam_messages) + ["ham"] * len(ham_messages)
})

# Okt 토크나이저 준비
okt = Okt()
def okt_tokenizer(text):
  return okt.morphs(text)

# TF-IDF 변환
vectorizer = TfidfVectorizer(tokenizer=okt_tokenizer)
X = vectorizer.fit_transform(df["message"])

print("전체 개수:", len(df))
print(df["label"].value_counts())
df.head()

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


전체 개수: 100
label
spam    50
ham     50
Name: count, dtype: int64


,message,label
0,축하합니다! 100만원에 당첨되셨습니다 지금 클릭 http://bit.ly/win123,spam
1,[Web발신] 국세청 환급금 조회 안내 http://tax-refund.xyz,spam
2,택배 배송 주소가 불분명합니다 정보 입력 http://delivery-check.com,spam
3,엄마 나 폰 고장나서 그런데 이 링크로 인증 좀 해줘 http://help-me.net,spam
4,고객님 카드가 해외에서 결제되었습니다 본인 아니면 클릭 http://card-sto...,spam


# 4. TF-IDF 변환 + 학습/시험 분할

In [4]:
# 입력(X)과 정답(y) 나누기
X = df["message"]
y = df["label"]

# 학습용 / 시험용 분할 (앞에서 배운 stratify, random_state 그대로)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# TF-IDF 변환 (Okt 토크나이저 연결)
vectorizer = TfidfVectorizer(tokenizer=okt_tokenizer)
X_train_vec = vectorizer.fit_transform(X_train)  # 학습용: 사전 만들고 + 변환
X_test_vec = vectorizer.transform(X_test)        # 시험용: 변환만

print("학습용 행렬 모양:", X_train_vec.shape)
print("시험용 행렬 모양:", X_test_vec.shape)
print("TF-IDF 차원 수:", X_train_vec.shape[1])

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


학습용 행렬 모양: (80, 403)
시험용 행렬 모양: (20, 403)
TF-IDF 차원 수: 403


In [5]:
# 1) 라벨을 숫자로: ham=0, spam=1
y_train_num = (y_train == "spam").astype(int).values
y_test_num = (y_test == "spam").astype(int).values

# 2) TF-IDF 결과를 텐서로 변환
#    .toarray() : 희소행렬 → 일반 배열
#    dtype 주의! 입력은 float, 라벨은 long(정수)
X_train_t = torch.tensor(X_train_vec.toarray(), dtype=torch.float32)
X_test_t  = torch.tensor(X_test_vec.toarray(),  dtype=torch.float32)
y_train_t = torch.tensor(y_train_num, dtype=torch.long)
y_test_t  = torch.tensor(y_test_num,  dtype=torch.long)

print("입력 텐서 모양:", X_train_t.shape)
print("라벨 텐서 모양:", y_train_t.shape)

입력 텐서 모양: torch.Size([80, 403])
라벨 텐서 모양: torch.Size([80])


# 5. MLP 모델

In [6]:
class SpamMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # 층 정의
        self.fc1 = nn.Linear(input_dim, 64)  # 입력층 → 은닉층(64개 뉴런)
        self.relu = nn.ReLU()                # 활성함수
        self.fc2 = nn.Linear(64, 2)          # 은닉층 → 출력층(2개: ham/spam)

    def forward(self, x):
        x = self.fc1(x)    # 1) 입력을 은닉층으로
        x = self.relu(x)   # 2) 활성함수 통과
        x = self.fc2(x)    # 3) 출력층으로 (최종 2개 값)
        return x

# 모델 생성 (input_dim은 TF-IDF 차원 수)
input_dim = X_train_t.shape[1]
model_pt = SpamMLP(input_dim)
print(model_pt)

SpamMLP(
  (fc1): Linear(in_features=403, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)


# 6. 손실함수와 옵티마이저

In [7]:
# 손실함수: 예측이 정답과 얼마나 다른지 재는 도구
criterion = nn.CrossEntropyLoss()

# 옵티마이저: 손실을 줄이는 방향으로 가중치를 수정하는 도구
optimizer = torch.optim.Adam(model_pt.parameters(), lr=0.01)

# 7. DataLoader 만들기

In [8]:
# 입력 텐서와 라벨 텐서를 하나로 묶기
train_dataset = TensorDataset(X_train_t, y_train_t)

# DataLoader: 데이터를 batch 단위로 섞어서 공급
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

print("배치 크기:", 16)
print("전체 학습 데이터:", len(train_dataset), "개")
print("한 에포크당 배치 수:", len(train_loader))

배치 크기: 16
전체 학습 데이터: 80 개
한 에포크당 배치 수: 5


In [9]:
num_epochs = 200  # 전체 데이터를 200번 반복 학습

print("모델 학습 시작...")
for epoch in range(num_epochs):
    for batch_X, batch_y in train_loader:
        # 순전파: 예측을 내본다
        outputs = model_pt(batch_X)
        loss = criterion(outputs, batch_y)  # 얼마나 틀렸나

        # 역전파 + 가중치 수정
        optimizer.zero_grad()  # 이전 기울기 초기화
        loss.backward()        # 고칠 방향 계산
        optimizer.step()       # 실제로 가중치 수정

    # 20 에포크마다 현재 손실 출력
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("모델 학습 완료.")

모델 학습 시작...
Epoch [20/200], Loss: 0.0002
Epoch [40/200], Loss: 0.0000
Epoch [60/200], Loss: 0.0000
Epoch [80/200], Loss: 0.0000
Epoch [100/200], Loss: 0.0000
Epoch [120/200], Loss: 0.0000
Epoch [140/200], Loss: 0.0000
Epoch [160/200], Loss: 0.0000
Epoch [180/200], Loss: 0.0000
Epoch [200/200], Loss: 0.0000
모델 학습 완료.


# 8. 평가

In [10]:
# 평가 모드로 전환 (학습이 아니라 예측만 할 거라는 선언)
model_pt.eval()

# 기울기 계산 끄기 (평가 땐 학습 안 하니까 — 메모리 절약, 속도 향상)
with torch.no_grad():
    test_outputs = model_pt(X_test_t)          # 시험용 데이터로 예측
    _, predicted = torch.max(test_outputs, 1)  # 두 점수 중 큰 쪽의 인덱스 = 예측 라벨

# 예측 결과를 다시 spam/ham 글자로 (보기 좋게)
pred_labels = ["spam" if p == 1 else "ham" for p in predicted]
true_labels = ["spam" if y == 1 else "ham" for y in y_test_t]

# 성능 리포트
print("=== PyTorch MLP 성능 ===")
print(classification_report(true_labels, pred_labels))
print("혼동행렬:")
print(confusion_matrix(true_labels, pred_labels))

=== PyTorch MLP 성능 ===
              precision    recall  f1-score   support

         ham       0.75      0.90      0.82        10
        spam       0.88      0.70      0.78        10

    accuracy                           0.80        20
   macro avg       0.81      0.80      0.80        20
weighted avg       0.81      0.80      0.80        20

혼동행렬:
[[9 1]
 [3 7]]


## 추가. baseline과 비교


In [11]:
from sklearn.linear_model import LogisticRegression

# baseline: 같은 TF-IDF 데이터로 로지스틱 회귀 학습
# (PyTorch용으로 toarray() 한 것 말고, 원래 희소행렬 X_train_vec 사용)
lr_model = LogisticRegression()
lr_model.fit(X_train_vec, y_train)  # y_train은 원래 "spam"/"ham" 글자 그대로

# 시험용 예측
lr_pred = lr_model.predict(X_test_vec)

print("=== 로지스틱 회귀(baseline) 성능 ===")
print(classification_report(y_test, lr_pred))
print("혼동행렬:")
print(confusion_matrix(y_test, lr_pred))

=== 로지스틱 회귀(baseline) 성능 ===
              precision    recall  f1-score   support

         ham       0.82      0.90      0.86        10
        spam       0.89      0.80      0.84        10

    accuracy                           0.85        20
   macro avg       0.85      0.85      0.85        20
weighted avg       0.85      0.85      0.85        20

혼동행렬:
[[9 1]
 [2 8]]


# 한국어 스팸(스미싱) 분류기 — 학습 정리

> 졸작(QR 보안 정보 서비스) 사전 워밍업 프로젝트
> 텍스트 분류 파이프라인을 처음부터 끝까지 한 번 관통하며 익힌 내용 정리

---

## 0. 이 프로젝트의 목적

- **목표**: 한국어 문자 메시지를 입력받아 `스팸(spam)` / `정상(ham)`으로 분류
- **진짜 목적**: 완벽한 성능이 아니라, **데이터 → 학습 → 예측 → 평가** 파이프라인을 끝까지 한 번 완주하는 것
- **졸작과의 연결**: QR이 디코딩한 URL을 텍스트로 보면, 똑같은 파이프라인에 그대로 태울 수 있음. "특징 추출 → 분류 → 점수화" 구조가 동일.

---

## 1. 전체 흐름 한눈에

```
데이터 직접 설계 (5:5, 헷갈리는 예시 포함)
   ↓
한국어 형태소 분석 (Okt)
   ↓
TF-IDF 벡터화
   ↓
baseline 모델 (로지스틱 회귀)
   ↓
데이터 누수 방지 (fit / transform 구분)
   ↓
평가 지표 이해 (precision / recall / F1, 혼동행렬)
   ↓
신경망 직접 구현 (층 · 활성함수 · 손실함수 · 학습 루프)
   ↓
과적합 진단
   ↓
baseline과 공정 비교
   ↓
보안 관점의 트레이드오프 판단 (recall 중시)
```

---

## 2. 데이터 직접 설계

### 핵심 결정
- **비율 5:5** — 연습용에선 균형을 맞춰야 모델이 한쪽으로 치우쳐 찍는 꼼수를 못 부리고 양쪽 패턴을 고루 학습. (단, **실제 세상은 스팸이 소수**라 불균형. 나중에 실데이터 붙이면 F1·recall이 중요해짐)
- **헷갈리는 예시 일부러 포함** — 예: "축하/당첨"이라는 같은 단어를 진짜 이벤트 안내(ham)와 가짜 당첨 사칭(spam) 양쪽에 넣음. 모델이 단어 하나가 아니라 **맥락(링크 유도, 개인정보 요구)**으로 판단하도록 유도.

### 주의점
- 메시지 순서와 라벨 순서가 어긋나면 안 됨.
- 정렬된 채로 앞/뒤를 자르면 한쪽 라벨만 시험용에 들어가는 문제 발생 → **섞기(shuffle) 필요**.

---

## 3. 한국어 형태소 분석 (Okt)

### 왜 필요한가
- 영어는 띄어쓰기로 단어가 깔끔하게 나뉘지만, 한국어는 "쿠폰을 / 쿠폰이 / 쿠폰은"이 전부 다른 단어로 취급됨.
- **형태소** 단위로 쪼개면 "쿠폰 + 을"처럼 조사를 분리해 같은 "쿠폰"으로 묶임.

### 코드
```python
!pip install konlpy   # Colab엔 기본 설치 안 됨 (PyTorch는 기본 설치됨)

from konlpy.tag import Okt
okt = Okt()

def okt_tokenizer(text):
    return okt.morphs(text)   # morphs = 형태소 단위로 쪼개기

print(okt_tokenizer("무료 쿠폰을 지금 받으세요"))
# → ['무료', '쿠폰', '을', '지금', '받으세요']
```

### 조사·문장부호는 빼야 하나?
- **문장부호**("!", "★")는 스팸이 더 자주 써서 **신호가 될 수 있음** → 빼면 정보 손실 가능.
- **조사**("을/를/이/가")는 양쪽에 고루 나와 변별력 낮음. 다만 **TF-IDF의 IDF가 자동으로 가중치를 낮춰줌** → 굳이 손으로 안 빼도 됨.
- **결론**: 지금은 다 넣고 진행. 나중에 "빼기 vs 넣기"를 F1으로 비교(ablation)하면 졸작 보고서의 좋은 근거가 됨.

---

## 4. TF-IDF 벡터화

### 개념
- 컴퓨터는 글자를 못 읽으니 문장을 숫자 배열로 변환해야 함.
- **TF (단어 빈도)**: 한 문장에서 그 단어가 얼마나 자주 나오나.
- **IDF (역문서 빈도)**: 그 단어가 전체 문장에서 얼마나 흔한가를 **반대로** 반영. 흔한 단어("은/는/이/가")는 가중치↓, 드문 단어("당첨/클릭")는 가중치↑.
- **TF × IDF** = "이 문장에서 자주 나오면서 + 다른 문장엔 잘 안 나오는" 특징적 단어가 높은 점수.

### 입력값이 실제로 뭔가 (중요)
- 사전에 단어가 N종류면, 각 문장은 **N칸짜리 숫자 배열**이 됨.
- 각 칸 = 사전의 특정 단어. 문장에 있으면 TF-IDF 값, 없으면 0.
- 한 문장엔 단어가 몇 개 안 들어가니 대부분 0 → **희소(sparse) 행렬**.

```python
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(tokenizer=okt_tokenizer)  # 한국어 토크나이저 연결
X_train_vec = vectorizer.fit_transform(X_train)  # 학습용: 사전 만들고(fit) + 변환
X_test_vec  = vectorizer.transform(X_test)       # 시험용: 변환만(transform)
```

### TF-IDF는 임베딩이 아니다
- TF-IDF = **빈도 기반**. 단어의 의미는 모름. (1세대)
- 임베딩(Word2Vec, BERT) = **의미 기반** 벡터. "왕 - 남자 + 여자 ≈ 여왕". (2~3세대)
- 헷갈리지 말 것. 지금 단계는 "토큰화 → TF-IDF 벡터화"이지 임베딩이 아님.

---

## 5. 데이터 누수 방지 — fit / transform 구분 (★중요 원칙)

### 핵심
- `fit` = "단어 사전 만들기" (데이터를 훑어 단어 목록·IDF를 정함) = **공부**
- `transform` = 그 사전으로 문장을 숫자로 **바꾸기만**
- **fit은 무조건 학습용에만!** 시험용엔 transform만.

### 왜?
- 시험용 데이터는 "미래에 들어올, 모르는 데이터"를 흉내 냄 → 모델 준비 과정에 끼면 안 됨.
- 시험용에도 fit하면 시험 정보가 학습에 새어 들어감 = **데이터 누수(data leakage)**.
- 시험지를 미리 본 셈이라 점수가 부풀려지고, 실제 서비스에선 성능이 뚝 떨어짐.

### 부수 효과
- 같은 사전을 쓰므로 학습용·시험용 행렬의 **열 개수(단어 종류 수)가 동일**해야 함.
- 시험용에만 있는 새 단어는 사전에 없으니 **무시됨** → 데이터를 키워야 사전이 풍부해짐.

---

## 6. baseline 모델 — 로지스틱 회귀

### 왜 baseline부터?
- "딥러닝부터"가 아니라 **"baseline부터"**. 모든 비교의 기준선.
- 단순/빠르고, 결과 해석이 쉬움(어떤 단어가 +/- 가중치인지 바로 확인 가능).

### 개념
- 각 단어마다 "스팸이면 +, 정상이면 -" 가중치를 학습.
- 문장의 단어 가중치를 다 더한 뒤 **시그모이드**로 0~1 확률로 변환. 0.5 기준 판정.

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_vec, y_train)   # 학습은 한 줄. 진짜 일은 앞의 전처리.
```

### 데이터 양과 확신도
- 데이터 20개 → 새 문자 스팸 확률이 57%, 43% 등 **50% 근처에서 애매** (운에 가까움).
- 데이터 100개 → 64.5%, 31% 등 **양극단으로 또렷해짐**.
- 교훈: **"데이터가 곧 성능"**.

---

## 7. 평가 지표 이해

### 왜 정확도(accuracy)만 보면 안 되나
- 데이터가 한쪽으로 쏠리면(예: ham이 6배), 무조건 ham만 찍어도 정확도 85%가 나옴 → 의미 없음.

### 4가지 지표
- **Precision (정밀도)**: 스팸이라 예측한 것 중 진짜 스팸 비율. (오경보를 줄이는 관점)
- **Recall (재현율)**: 실제 스팸 중 잡아낸 비율. (놓침을 줄이는 관점)
- **F1**: precision과 recall의 조화평균. 둘의 균형.
- **혼동행렬(confusion matrix)**: 무엇을 무엇으로 착각했는지 2×2 표로.

### 혼동행렬 읽는 법
```
            예측 ham   예측 spam
실제 ham       9          1       ← 정상을 스팸으로 오해 1개 (A)
실제 spam      3          7       ← 스팸을 놓침 3개 (B)
```
- 대각선(9, 7)이 정답. 나머지가 오류.

---

## 8. 신경망 직접 구현 (PyTorch MLP)

### sklearn vs PyTorch — 결정적 차이
- sklearn: `model.fit()` 한 줄 안에서 **알아서 다 함** (블랙박스).
- PyTorch: 학습 과정을 **직접 루프로 돌림** → 각 단계를 보고 만질 수 있음.

### MLP 구조
- 로지스틱 회귀 = 층 1개. MLP = 사이에 **은닉층(hidden layer)** 추가 → 더 복잡한 패턴 학습.
- 구조: `입력층(TF-IDF 차원) → 은닉층(64) → ReLU → 출력층(2: ham/spam)`

```python
import torch
import torch.nn as nn

class SpamMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)  # 입력 → 은닉
        self.relu = nn.ReLU()                # 활성함수
        self.fc2 = nn.Linear(64, 2)          # 은닉 → 출력(2개)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
```

### 활성함수(ReLU)는 왜 필요한가 (★시험 단골)
- 활성함수 없이 선형층만 쌓으면 → 결국 **하나의 선형 변환**으로 합쳐져 직선밖에 못 그림.
- ReLU가 **비선형성**을 넣어줘야 곡선 같은 복잡한 결정 경계를 학습 가능.

### 텐서 변환 — 타입 주의 (★시험 단골)
- **입력(X)은 `float32`**: TF-IDF는 0.27 같은 **소수(연속 실수)**라 실수형 필요.
- **라벨(y)은 `long`(정수)**: 클래스 번호(0=ham, 1=spam)는 **딱 떨어지는 정수**. + `CrossEntropyLoss`가 long을 요구.
- 라벨을 숫자로 바꾸는 이유: 신경망은 **글자를 계산할 수 없음**. 손실 계산(뺄셈·미분)을 하려면 숫자여야 함.

```python
y_train_num = (y_train == "spam").astype(int).values
X_train_t = torch.tensor(X_train_vec.toarray(), dtype=torch.float32)
y_train_t = torch.tensor(y_train_num, dtype=torch.long)
```

### 손실함수 & 옵티마이저
```python
criterion = nn.CrossEntropyLoss()                                  # 분류용 손실함수
optimizer = torch.optim.Adam(model_pt.parameters(), lr=0.01)       # 가중치 수정 도구
```
- **CrossEntropyLoss vs BCELoss**: 출력을 2개(ham/spam 점수)로 설계 → 다중클래스용 CrossEntropyLoss. 출력 1개(스팸 확률)였다면 BCELoss. **출력 구조가 손실함수를 결정**.
- **학습률(lr)**: "한 번에 얼마나 크게 고칠지"의 보폭.
  - 너무 크면 → 최저점을 건너뛰며 발산/진동, 수렴 어려움.
  - 너무 작으면 → 조금씩만 이동, 수렴까지 너무 오래 걸림.

### DataLoader (batch 공급)
```python
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
```
- **shuffle=True**: 매 에포크 순서를 섞음 → 모델이 "데이터 순서"를 외우는 걸 방지.

### 학습 루프 — 5단계 반복
```python
num_epochs = 200
for epoch in range(num_epochs):
    for batch_X, batch_y in train_loader:
        outputs = model_pt(batch_X)            # ① 순전파: 일단 풀어본다
        loss = criterion(outputs, batch_y)     # ② 손실: 얼마나 틀렸나
        optimizer.zero_grad()                  # ③ 기울기 초기화 (이전 메모 지우기)
        loss.backward()                        # ④ 역전파: 고칠 방향 계산
        optimizer.step()                       # ⑤ 가중치 수정: 실제로 조금 고침
```
- **epoch**: 전체 데이터를 한 번 다 훑는 것. 반복해야 패턴을 익힘.
- **zero_grad() 꼭 필요**: PyTorch는 기울기를 누적하므로 매번 초기화 안 하면 과거 값이 쌓여 잘못 학습됨. (입문자 단골 실수)

---

## 9. 과적합(overfitting) 진단

### 증상
- 학습 데이터 Loss = 0.0000 (100% 정답)인데 시험 데이터 정확도 = 80%.
- 이 **간극**이 과적합의 증거.

### 원인
- 데이터(80개)는 적은데 모델(은닉 64뉴런)은 표현력이 큼 → 패턴을 "이해"하지 않고 "암기".
- 200 에포크는 이 작은 데이터엔 과함(20 에포크 만에 이미 Loss 0 도달).

### 비유
- 기출 100문제를 달달 외운 학생 → 기출은 만점, 새 문제(실전)엔 약함.

---

## 10. baseline과 공정 비교

### 결과 (같은 데이터 100개, 같은 분할 random_state=42)

| 항목 | 로지스틱 회귀 (baseline) | PyTorch MLP |
|---|---|---|
| 정확도 | **0.85** | 0.80 |
| spam recall | **0.80** (8/10) | 0.70 (7/10) |
| spam f1 | **0.84** | 0.78 |
| 스팸 놓침(B) | **2개** | 3개 |

### 결론: baseline이 이김
- **"복잡한 모델 ≠ 항상 좋은 모델"**.
- 데이터가 적으면 단순 모델이 신경망을 이기는 경우가 흔함. 신경망은 데이터가 많아야 빛남.
- baseline을 먼저 세우고, 그걸 **이겨야** 신경망을 쓸 이유가 생김.
- MLP가 쓸모없는 게 아니라, "지금 데이터에선" 진 것. 데이터를 수천 개로 키우면 역전 가능성.

### 공정한 실험의 핵심
- 같은 데이터, 같은 분할, 같은 TF-IDF 사전. **모델만 바꿈** → 차이는 순수하게 모델 차이.

---

## 11. 보안 관점의 트레이드오프 — recall 중시

### 두 실수 중 무엇이 더 위험한가
- (A) 정상을 스팸으로 오경보 → 살짝 짜증, **피해는 없음**.
- (B) 진짜 스팸을 놓침 → 악성 링크 클릭, **금전 피해·개인정보 유출**. ← **더 위험**

### 그래서
- 보안/탐지 분야의 기본 철학: **"의심스러우면 일단 경고"**.
- B를 줄인다 = **recall(재현율)을 높인다**.
- 트레이드오프: recall↑ 하면 precision↓ (오경보 A가 늘어남). B가 더 위험하니 recall 쪽으로 기우는 게 맞음.
- 졸작 보고서에 "왜 재현율 중심으로 튜닝했는가"를 보안 관점으로 설명하면 강력한 설계 근거.

---

## 12. 다음에 해볼 것 (메모)

- [ ] **모델 해석**: 로지스틱 회귀 가중치를 꺼내 "어떤 단어가 스팸 신호인지" 확인 → 졸작의 "왜 이 점수인지 설명" 기능과 직결.
- [ ] **recall 높이기 실험**: 판정 임계값 조정, 클래스 가중치(class_weight) 등으로 B(스팸 놓침) 줄이기.
- [ ] **전처리 ablation**: 조사/문장부호 빼기 vs 넣기를 F1으로 비교.
- [ ] **데이터 키우기**: 수천 개로 늘려 MLP 재대결 (신경망이 역전하는지).
- [ ] **공개 데이터 연결**: AI Hub, 공공데이터포털의 스미싱/스팸 데이터.
- [ ] **GitHub 정리**: 노트북을 레포로 올려 포트폴리오화.

---

## 한 줄 요약

> 작은 데이터로 텍스트 분류 파이프라인 전체(설계 → 형태소 분석 → 벡터화 → baseline → 신경망 → 평가 → 비교)를 완주했고,
> **"baseline부터, 데이터가 곧 성능, 복잡한 모델이 항상 이기는 건 아니다"**라는 핵심 교훈을 직접 확인함.
> 이 구조는 졸작 QR 안전도 점수화 모듈의 축소판이다.